# Notebook 6 — National parametric benchmarks: Cox PH + Weibull AFT

Fits the two standard survival-regression baselines on the **same national
first-hit-poor dataset and the same seed-42 80/20 split** as the RSF
(`train_national_rsf.ipynb`), so all three models are directly comparable on
C-index, integrated Brier score, and time-dependent AUC.

**Inputs** (all produced by Notebook 4, `build_national_rsf_dataset.ipynb`):
- `us_parametric_data_clean.csv` — the drop-first (full-rank) design matrix Notebook 4
  writes specifically for parametric models (no dummy-variable trap).
- `us_rsf_data_clean.csv` — keys only, to assert both matrices hold the same bridges
  in the same row order (which makes the seed-42 split land on identical bridges).
- `us_rsf_metrics.json` — the RSF benchmark numbers + its IBS time grid (reused here
  so the Brier scores integrate over the same horizons).

**Outputs:** `us_parametric_metrics.json`, `us_cox_hazard_ratios.csv`,
`us_aft_coefficients.csv`.

**Method notes**
- lifelines cannot accept NaN, so features are filled with **train-only medians**,
  with the missingness kept as explicit 0/1 flags (one per distinct pattern — the
  18 climate columns share one mask; the `years_since_reconstruction` pattern is
  skipped because it duplicates `ever_reconstructed == 0`).
- Both fitters get a small ridge penalty (`penalizer = 0.01`): the design still has
  near-collinear dummy blocks, and at n ≈ 1M the shrinkage bias is negligible.
- The AFT arm standardizes continuous features with train statistics (its likelihood
  is poorly conditioned on raw scales, unlike `CoxPHFitter`, which normalizes
  internally) — so continuous AFT coefficients in the CSV are per standard deviation.
- What the comparison means: Cox assumes proportional hazards and linear effects;
  Weibull AFT additionally assumes a Weibull baseline. Their gap to the RSF measures
  how much national deck/sub/superstructure deterioration depends on interactions and
  non-linearities that a linear index cannot express — in exchange, Cox/AFT give
  interpretable hazard ratios / time ratios per feature.


In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
import json
import time
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.model_selection import train_test_split
from sksurv.util import Surv
from sksurv.metrics import (concordance_index_censored, integrated_brier_score,
                            cumulative_dynamic_auc)
from lifelines import CoxPHFitter, WeibullAFTFitter

# Inputs (Notebook 4 writes both matrices from the same Option-A table, same row order)
DATA_PARAM  = Path("us_parametric_data_clean.csv")   # drop-first design, full rank
DATA_RSF    = Path("us_rsf_data_clean.csv")          # keys only — proves split parity
RSF_METRICS = Path("us_rsf_metrics.json")            # benchmark + shared IBS time grid

KEYS = ["STATE_FIPS", "STRUCTURE_NUMBER_008"]   # structure # is unique only WITHIN a state

SMOKE_TEST = False   # True -> 20k-bridge subsample; outputs get a _smoke suffix

# Small L2 ridge on both fitters: the design still carries near-collinear blocks
# (dummy groups + missingness flags), and at n ~ 1M the shrinkage bias is negligible.
PENALIZER = 0.01

AUC_HORIZONS = [25.0, 50.0, 75.0]   # years — the RSF evaluation's horizons

def out_path(name):
    """Output path; smoke runs get a _smoke suffix so real artifacts are never clobbered."""
    p = Path(name)
    return p.with_stem(p.stem + "_smoke") if SMOKE_TEST else p

OUT_METRICS  = out_path("us_parametric_metrics.json")
OUT_COX_HR   = out_path("us_cox_hazard_ratios.csv")
OUT_AFT_COEF = out_path("us_aft_coefficients.csv")

if not DATA_PARAM.exists():
    raise FileNotFoundError(
        f"{DATA_PARAM} not found — it is produced by build_national_rsf_dataset.ipynb (Notebook 4).")

print(f"config OK — smoke={SMOKE_TEST}, penalizer={PENALIZER}")


In [ ]:
# ── Typed load (mirrors train_national_rsf cell 3) + split-parity check ───────
cols = pd.read_csv(DATA_PARAM, nrows=0).columns
dtypes = {c: "float32" for c in cols}
dtypes.update({k: str for k in KEYS})
dtypes["event"] = "int8"
df = pd.read_csv(DATA_PARAM, dtype=dtypes)
for k in KEYS:
    df[k] = df[k].str.strip()

# Same bridges in the same row order as the RSF matrix -> the seed-42 split below
# lands on exactly the bridges the RSF was tested on, so metrics are comparable.
rsf_keys = pd.read_csv(DATA_RSF, usecols=KEYS, dtype=str)[KEYS]
for k in KEYS:
    rsf_keys[k] = rsf_keys[k].str.strip()
assert len(df) == len(rsf_keys) and (df[KEYS].to_numpy() == rsf_keys.to_numpy()).all(), \
    "parametric matrix rows do not line up with the RSF matrix — split would not be comparable"
del rsf_keys

if SMOKE_TEST and len(df) > 20_000:
    df = df.sample(n=20_000, random_state=42).reset_index(drop=True)
    print(f"SMOKE_TEST: subsampled to {len(df):,} bridges")

assert (df["time"] > 0).all(), "non-positive survival times — Notebook 4 should have dropped these"

y   = Surv.from_arrays(event=df["event"].astype(bool).to_numpy(),
                       time=df["time"].astype("float64").to_numpy())
ids = df[KEYS].copy()

# bridge_age == time in this file, so dropping it is mandatory (same as the RSF notebook).
X = df.drop(columns=["event", "time", "bridge_age"] + KEYS)

leak = [c for c in X.columns if "_COND_" in c or c == "OPEN_CLOSED_POSTED_041"]
assert not leak, f"leakage columns present: {leak}"
assert X.select_dtypes(exclude="number").shape[1] == 0, "non-numeric feature columns in X"

del df
print(f"X: {X.shape[0]:,} bridges x {X.shape[1]} features   "
      f"event rate: {y['event'].mean()*100:.1f}%   NaN share: {X.isna().mean().mean()*100:.1f}%")


In [ ]:
# ── Missingness flags + identical 80/20 split + train-median imputation ───────
# lifelines cannot take NaN. Keep the missingness signal as explicit 0/1 flags
# (one per distinct pattern), then fill with TRAIN medians so nothing about the
# test rows leaks into the imputation.
na_cols = X.columns[X.isna().any()].tolist()

never_reconstructed = (X["ever_reconstructed"] == 0).to_numpy().astype("int8")
seen, flags = set(), {}
for c in na_cols:
    m = X[c].isna().to_numpy()
    key = m.tobytes()
    if key in seen:
        continue                      # duplicate pattern (e.g. the climate block) -> one flag
    seen.add(key)
    f = m.astype("int8")
    if (f == never_reconstructed).all():
        continue                      # already encoded by ever_reconstructed
    flags[c + "_missing"] = f
X = pd.concat([X, pd.DataFrame(flags, index=X.index)], axis=1)
print(f"{len(na_cols)} columns with NaN -> {len(flags)} distinct missingness flags")

X_train, X_test, y_train, y_test, ids_train, ids_test = train_test_split(
    X, y, ids, test_size=0.2, random_state=42)

med = X_train[na_cols].median()
X_train = X_train.fillna(med)
X_test  = X_test.fillna(med)
assert not X_train.isna().any().any() and not X_test.isna().any().any()

# Columns constant in train carry no information and break the optimizer's scaling.
const = X_train.columns[(X_train.max(axis=0) == X_train.min(axis=0)).to_numpy()].tolist()
if const:
    X_train, X_test = X_train.drop(columns=const), X_test.drop(columns=const)
    print(f"dropped {len(const)} train-constant columns: {const[:8]}{'...' if len(const) > 8 else ''}")

print(f"train: {len(X_train):,}   test: {len(X_test):,}   features: {X_train.shape[1]}")


In [ ]:
# ── Cox proportional hazards ──────────────────────────────────────────────────
train_df = X_train.assign(time=y_train["time"], event=y_train["event"].astype("int8"))

t0 = time.time()
cph = CoxPHFitter(penalizer=PENALIZER)
cph.fit(train_df, duration_col="time", event_col="event", show_progress=True)
cox_fit_s = time.time() - t0

cox_risk = cph.predict_partial_hazard(X_test).to_numpy()   # higher = riskier
cox_ci = concordance_index_censored(y_test["event"], y_test["time"], cox_risk)[0]
print(f"\nCox PH: test C-index {cox_ci:.4f}   fit {cox_fit_s:.0f}s   "
      f"({len(cph.params_)} coefficients)")

hr = cph.summary[["coef", "exp(coef)", "exp(coef) lower 95%", "exp(coef) upper 95%", "p"]]
hr = hr.sort_values("coef", key=np.abs, ascending=False)
hr.to_csv(OUT_COX_HR)
print(f"hazard ratios -> {OUT_COX_HR}")
hr.head(10)


In [ ]:
# ── Weibull AFT (same design, same split) ─────────────────────────────────────
# The AFT likelihood is poorly conditioned on raw feature scales (ADT ~ 1e5 next
# to 0/1 dummies) — an unscaled fit hits scipy's iteration cap. CoxPHFitter
# normalizes internally; the AFT fitters do not, so standardize the continuous
# columns with TRAIN statistics. Saved continuous-feature coefficients are per-SD.
is_binary = ((X_train == 0) | (X_train == 1)).all()
cont = X_train.columns[~is_binary]
mu, sd = X_train[cont].mean(), X_train[cont].std()
Xtr_aft, Xte_aft = X_train.copy(), X_test.copy()
Xtr_aft[cont] = (Xtr_aft[cont] - mu) / sd
Xte_aft[cont] = (Xte_aft[cont] - mu) / sd
train_df_aft = Xtr_aft.assign(time=y_train["time"], event=y_train["event"].astype("int8"))

t0 = time.time()
aft = WeibullAFTFitter(penalizer=PENALIZER)
# scipy's SLSQP reaches this likelihood's optimum but needs more than its default
# 200-iteration cap (verified on a 20k smoke subsample: caps at 200, converges under 2000)
aft.fit(train_df_aft, duration_col="time", event_col="event", show_progress=True,
        fit_options={"maxiter": 2000})
aft_fit_s = time.time() - t0

# AFT predicts survival time: longer predicted survival = lower risk -> negate.
aft_median = aft.predict_median(Xte_aft).to_numpy(dtype="float64")
aft_ci = concordance_index_censored(y_test["event"], y_test["time"], -aft_median)[0]
print(f"\nWeibull AFT: test C-index {aft_ci:.4f}   fit {aft_fit_s:.0f}s   "
      f"({len(cont)} continuous features standardized)")

aft.summary.to_csv(OUT_AFT_COEF)
print(f"coefficients -> {OUT_AFT_COEF}")


In [ ]:
# ── IBS + time-dependent AUC on the RSF's own evaluation grid ─────────────────
grid = np.asarray(json.loads(RSF_METRICS.read_text())["ibs_time_grid"], dtype=float)
lo = max(float(y_test["time"].min()), float(y_train["time"].min()))
hi = min(float(y_test["time"].max()), float(y_train["time"].max()))
grid = grid[(grid > lo) & (grid < hi)]

def surv_matrix(model, X, times):
    """(n, n_times) survival probabilities, chunked to keep float64 blocks small."""
    out = []
    for i in range(0, len(X), 50_000):
        out.append(model.predict_survival_function(X.iloc[i:i + 50_000], times=times).to_numpy().T)
    return np.vstack(out)

ibs, auc = {}, {}
for name, model, Xte in [("cox_ph", cph, X_test), ("weibull_aft", aft, Xte_aft)]:
    S = surv_matrix(model, Xte, grid)
    ibs[name] = float(integrated_brier_score(y_train, y_test, S, grid))
    risk_h = 1.0 - surv_matrix(model, Xte, np.asarray(AUC_HORIZONS))
    a, mean_a = cumulative_dynamic_auc(y_train, y_test, risk_h, AUC_HORIZONS)
    auc[name] = {"by_horizon": {f"{int(h)}yr": round(float(v), 4) for h, v in zip(AUC_HORIZONS, a)},
                 "mean": round(float(mean_a), 4)}
    print(f"{name}: IBS {ibs[name]:.4f}   AUC {auc[name]['by_horizon']}   mean {auc[name]['mean']:.4f}")


In [ ]:
# ── Comparison vs the RSF benchmark + persist metrics ─────────────────────────
rsf = json.loads(RSF_METRICS.read_text())
rows = [
    ("RSF (train_national_rsf)", rsf["c_index_test"], rsf["integrated_brier_score"],
     rsf["auc_mean"], rsf["fit_seconds"]),
    ("Cox PH", cox_ci, ibs["cox_ph"], auc["cox_ph"]["mean"], cox_fit_s),
    ("Weibull AFT", aft_ci, ibs["weibull_aft"], auc["weibull_aft"]["mean"], aft_fit_s),
]
print(f"{'model':<26} {'C-index':>8} {'IBS':>7} {'mean AUC':>9} {'fit (s)':>8}")
for n, c, b, a, s in rows:
    print(f"{n:<26} {c:>8.4f} {b:>7.4f} {a:>9.4f} {s:>8.0f}")

metrics = {
    "generated_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "smoke_test": SMOKE_TEST,
    "n_bridges": int(len(X_train) + len(X_test)),
    "n_train": int(len(X_train)),
    "n_test": int(len(X_test)),
    "n_features": int(X_train.shape[1]),
    "penalizer": PENALIZER,
    "imputation": "train-median + deduplicated 0/1 missingness flags",
    "cox_ph": {
        "c_index_test": round(float(cox_ci), 4),
        "integrated_brier_score": round(ibs["cox_ph"], 4),
        "auc_by_horizon": auc["cox_ph"]["by_horizon"],
        "auc_mean": auc["cox_ph"]["mean"],
        "fit_seconds": round(cox_fit_s, 1),
    },
    "weibull_aft": {
        "c_index_test": round(float(aft_ci), 4),
        "integrated_brier_score": round(ibs["weibull_aft"], 4),
        "auc_by_horizon": auc["weibull_aft"]["by_horizon"],
        "auc_mean": auc["weibull_aft"]["mean"],
        "fit_seconds": round(aft_fit_s, 1),
    },
    "rsf_reference": {
        "c_index_test": rsf["c_index_test"],
        "integrated_brier_score": rsf["integrated_brier_score"],
        "auc_mean": rsf["auc_mean"],
        "source": str(RSF_METRICS),
    },
}
OUT_METRICS.write_text(json.dumps(metrics, indent=2))
print(f"\nmetrics -> {OUT_METRICS}")


In [ ]:
# ── Output verification ───────────────────────────────────────────────────────
m = json.loads(OUT_METRICS.read_text())
for k in ("cox_ph", "weibull_aft"):
    assert 0.5 < m[k]["c_index_test"] < 1.0, (k, m[k])
    assert 0.0 < m[k]["integrated_brier_score"] < 0.25, (k, m[k])
    assert 0.5 < m[k]["auc_mean"] <= 1.0, (k, m[k])
hr_csv = pd.read_csv(OUT_COX_HR, index_col=0)
assert len(hr_csv) == X_train.shape[1] and {"coef", "exp(coef)", "p"} <= set(hr_csv.columns)
assert pd.read_csv(OUT_AFT_COEF).shape[0] >= X_train.shape[1]
print("VERIFICATION PASSED —",
      f"Cox C {m['cox_ph']['c_index_test']:.4f},",
      f"AFT C {m['weibull_aft']['c_index_test']:.4f},",
      f"RSF reference {m['rsf_reference']['c_index_test']:.4f}")
